# Syllabus-Grounded Contextual Biasing — full study on Kaggle

Kaggle suits this workload better than free Colab: **9-hour GPU sessions**, 30 GPU-hours
a week, and — the part that matters — `/kaggle/working` is saved as notebook output, so a
session ending does not destroy the decode cache.

## Settings to change before running (right-hand panel)

1. **Accelerator → GPU T4 x2** (or P100).
2. **Internet → On.** Needed for `pip`, `git clone` and the corpus download. Kaggle
   requires a phone-verified account to enable it.

## Reading the log

Every cell prints a banner when it starts and when it finishes, with a clock time and how
long it took:

```
========================================================================
[14:02:11] CELL 8/13  START  — Corpus + self-tests
========================================================================
...
[14:17:40] CELL 8/13  DONE   — Corpus + self-tests   (15m 29s)
```

So in a background run you can always tell which stage is live and which are finished.
Long steps also print a heartbeat line every ~30 seconds so you can see they are alive.

## Running it

**`Save Version → Save & Run All`** (the black button, top right) runs the whole notebook
on Kaggle's machines with the browser closed. The toolbar's `Run All` is the interactive
version and needs the tab alive.

**Session 1** runs cells 1–12 and stops. The last cell (Tier 3) is deliberately left as
Markdown so it is skipped — see the note above it.

**To resume in a later session,** attach this notebook's own output via *Add Data →
Notebook Output*. Cell 2 restores the decode cache from it, so finished work is never
repeated. Bulky regenerable things (the archive, the cut audio) live in `/kaggle/temp`,
which is wiped; the cache is tarred into `/kaggle/working/checkpoints/` after every stage.

## Two rules that keep the results valid

1. **One results table, one platform.** GPU float16 and CPU int8 produce different
   hypotheses. `device` and `compute_type` are part of the cache key so they cannot mix
   inside a cache, but do not put a CPU row and a GPU row in the same table.
2. **The pilots belong to the platform that reports the numbers**, so cell 9 re-runs them
   here. They cost about fifteen minutes on a GPU.

In [ ]:
# ============================ CELL 1: environment check ======================
import datetime, time, sys

_STARTS = {}
TOTAL = 13

def stage(n, title, end=False):
    """Print a start/finish banner so a flat background log stays readable."""
    now = time.time()
    ts = datetime.datetime.now().strftime('%H:%M:%S')
    bar = '=' * 72
    if end:
        el = now - _STARTS.get(n, now)
        took = f'{int(el // 60)}m {int(el % 60)}s'
        print(f'\n{bar}\n[{ts}] CELL {n}/{TOTAL}  DONE   — {title}   ({took})\n{bar}',
              flush=True)
    else:
        _STARTS[n] = now
        print(f'\n{bar}\n[{ts}] CELL {n}/{TOTAL}  START  — {title}\n{bar}', flush=True)

stage(1, 'Environment check')

!nvidia-smi
import socket
try:
    socket.create_connection(('pypi.org', 443), timeout=6).close()
    print('internet: ON')
except OSError:
    raise SystemExit('internet is OFF — enable it in Settings, or nothing below works')

stage(1, 'Environment check', end=True)

In [ ]:
# ============ CELL 2: layout, code, and restore from the last session ========
# /kaggle/working  is saved as notebook output  -> repo, runs/, report/, checkpoints/
# /kaggle/temp     is scratch, wiped on exit    -> archive, cut audio, live cache
stage(2, 'Layout + restore previous cache')

import os, glob, subprocess, pathlib, shutil

REPO = '/kaggle/working/FYP'
SCRATCH = '/kaggle/temp'
CKPT = pathlib.Path('/kaggle/working/checkpoints')
CKPT.mkdir(parents=True, exist_ok=True)

if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/meet244/FYP.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=False)
os.chdir(REPO)

# Keep the audio and the live cache out of the saved output.
for name in ('data', 'cache'):
    target = f'{SCRATCH}/{name}'
    os.makedirs(target, exist_ok=True)
    if not os.path.islink(name):
        shutil.rmtree(name, ignore_errors=True)
        os.symlink(target, name)

os.environ['PYTHONPATH'] = 'src'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Restore the newest checkpoint from an attached previous output, if there is one.
cands = sorted(glob.glob('/kaggle/input/*/checkpoints/*.tar.gz')
               + glob.glob('/kaggle/input/*/*/checkpoints/*.tar.gz'))
for tgz in cands:
    print('restoring', tgz)
    subprocess.run(['tar', '-xzf', tgz], check=False)
N_RESTORED = len(glob.glob('cache/asr/*/*/*.json'))
if cands:
    print(f'\n>>> {N_RESTORED} cached decodes restored — these will NOT be recomputed')
else:
    print('\n>>> no previous checkpoint attached; starting from scratch.')
    print('>>> If you meant to resume, stop now and attach the previous')
    print('>>> notebook output via Add Data -> Notebook Output.')
print('cwd:', os.getcwd())

stage(2, 'Layout + restore previous cache', end=True)

In [ ]:
# =================== CELL 3: live-output and checkpoint helpers ==============
stage(3, 'Helpers')

import subprocess, sys, time, glob, os

def run(*args, heartbeat=30):
    """Run a harness script, streaming its output live.

    Progress bars are collapsed into one heartbeat line every `heartbeat` seconds:
    without that a two-hour stage would print nothing until it finished, which is
    useless when the only thing you can see is a log.
    """
    t0 = time.time()
    last_beat = 0.0
    tail = []
    print(f'$ python {" ".join(args)}', flush=True)
    p = subprocess.Popen([sys.executable, *args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, 'PYTHONPATH': 'src'})
    for line in p.stdout:
        line = line.rstrip()
        if not line:
            continue
        tail.append(line)
        del tail[:-40]
        is_progress = ('utt/s' in line or 'it/s]' in line or 's/utt' in line
                       or 'it/s' in line)
        if is_progress:
            now = time.time()
            if now - last_beat >= heartbeat:
                last_beat = now
                print(f'    ... {int((now - t0) // 60)}m elapsed | {line[-90:]}',
                      flush=True)
        else:
            print(line, flush=True)
    p.wait()
    el = time.time() - t0
    print(f'  [{args[0]} finished in {int(el // 60)}m {int(el % 60)}s]', flush=True)
    if p.returncode:
        print('\n'.join(tail))
        raise RuntimeError(f'{args[0]} failed with exit code {p.returncode}')

def checkpoint(tag=''):
    """Snapshot the cache and results into the saved output after every stage.

    The decode cache is the expensive artefact. Tarring it here means a session that
    dies mid-matrix costs one condition, not the whole run.
    """
    subprocess.run(['tar', '-czf', str(CKPT / 'asr_cache.tar.gz'), 'cache/asr'],
                   check=False)
    subprocess.run(['tar', '-czf', str(CKPT / 'runs_report.tar.gz'),
                    'runs', 'report', 'configs', 'data/manifests'], check=False)
    n = len(glob.glob('cache/asr/*/*/*.json'))
    sz = sum(f.stat().st_size for f in CKPT.glob('*.tar.gz')) / 1e6
    print(f'>>> [checkpoint: {tag}] {n} decodes cached, {sz:.0f} MB saved to '
          f'/kaggle/working', flush=True)

print('helpers ready')
stage(3, 'Helpers', end=True)

In [ ]:
# ============================ CELL 4: dependencies ===========================
# requirements.txt pins CPU-oriented versions (numpy<2 etc.) that fight Kaggle's
# preinstalled torch, so install the minimum set and leave torch alone.
stage(4, 'Dependencies')

!pip install -q faster-whisper==1.1.1 "ctranslate2==4.5.0" jiwer==3.0.4 \
    rapidfuzz indic-transliteration sentence-transformers soundfile librosa \
    pyyaml matplotlib 2>&1 | tail -2
!python -c "import ctranslate2; print('ctranslate2', ctranslate2.__version__)"

stage(4, 'Dependencies', end=True)

In [ ]:
# ================= CELL 5: make cuDNN visible, prove the GPU works ===========
# CTranslate2 loads cuDNN from the NVIDIA pip wheels, not the image's default library
# path. Symptom when it cannot: 'Unable to load libcudnn_ops.so.9'.
stage(5, 'GPU / cuDNN check')

import site, glob, os, subprocess, sys
libdirs = {p for sp in site.getsitepackages() + [site.getusersitepackages()]
           for p in glob.glob(os.path.join(sp, 'nvidia', '*', 'lib'))}
os.environ['LD_LIBRARY_PATH'] = (':'.join(sorted(libdirs)) + ':'
                                 + os.environ.get('LD_LIBRARY_PATH', ''))
print(f'{len(libdirs)} nvidia wheel lib dirs on the path')

smoke = ("from faster_whisper import WhisperModel\n"
         "import numpy as np, soundfile as sf\n"
         "sf.write('/tmp/probe.wav', np.zeros(16000, dtype='float32'), 16000)\n"
         "m = WhisperModel('tiny', device='cuda', compute_type='float16')\n"
         "list(m.transcribe('/tmp/probe.wav', language='hi')[0])\n"
         "print('GPU decode OK')\n")
p = subprocess.run([sys.executable, '-c', smoke], capture_output=True, text=True,
                   env=os.environ)
print(p.stdout or p.stderr[-2000:])
if 'GPU decode OK' not in p.stdout:
    raise SystemExit('GPU decode failed — try: '
                     '!pip install -q nvidia-cudnn-cu12 nvidia-cublas-cu12, then re-run')

stage(5, 'GPU / cuDNN check', end=True)

In [ ]:
# ==================== CELL 6: point the frozen config at the GPU =============
# size, compute_type and device are part of the ASR cache key, so this cannot silently
# mix GPU results with CPU decodes. large-v3 is the §4.2 pilot's choice and is
# affordable on a GPU, which retires the 'knowingly weakened baseline' threat (§10).
stage(6, 'Config -> GPU')

import re, pathlib
p = pathlib.Path('configs/config.yaml'); t = p.read_text()
t = re.sub(r'^(\s*size:).*$', r'\1 large-v3              # §4.2 pilot decision', t, count=1, flags=re.M)
t = re.sub(r'^(\s*compute_type:).*$', r'\1 float16       # GPU precision', t, count=1, flags=re.M)
t = re.sub(r'^(\s*device:).*$', r'\1 cuda               # Kaggle T4', t, count=1, flags=re.M)
p.write_text(t)
!sed -n '/^model:/,/^decode:/p' configs/config.yaml

stage(6, 'Config -> GPU', end=True)

In [ ]:
# ================= CELL 7: corpus download, cut, refine, freeze ==============
# ~15 min. Re-run every session, because /kaggle/temp is wiped. Re-cutting is
# deterministic and the cache is keyed by utterance id, not file path, so restored
# decodes still hit after the audio is rebuilt.
stage(7, 'Corpus preparation')

import os
if not os.path.exists('data/raw/slr104/test/transcripts/text'):
    !mkdir -p data/raw/slr104
    !cd data/raw/slr104 && curl -sL -C - -o Hindi-English_test.tar.gz \
        https://openslr.trmal.net/resources/104/Hindi-English_test.tar.gz \
        && tar -xzf Hindi-English_test.tar.gz
run('src/prepare_slr104.py')
# The distributed segments are whole-second windows and are not usable as shipped;
# see report/01_dataset_and_harness.md §2.
run('src/refine_segments.py', '--radius', '2.5', '--lam', '2.0')
run('src/make_tiers.py', '--force')
run('src/build_syllabus.py')

stage(7, 'Corpus preparation', end=True)

In [ ]:
# ============================ CELL 8: self-tests =============================
# Seconds, no GPU. Checks the scoring, correction, gating and guard logic against
# hand-worked examples, then runs the whole condition matrix against a stub decoder.
# Never skip this before a multi-hour run.
stage(8, 'Self-tests')

run('src/selftest.py')
run('src/selftest_pipeline.py')

stage(8, 'Self-tests', end=True)

In [ ]:
# ==================== CELL 9: pilots — §4.3 language, §4.2 model =============
# ~15 min. Decides, with evidence rather than assumption, which language setting and
# which model the rest of the study uses, then writes both into the frozen config.
stage(9, 'Pilots (language, model)')

run('src/pilots.py', 'language', '--tier', 'tier1')
run('src/pilots.py', 'model', '--tier', 'tier1')
run('src/apply_pilot_decisions.py')
checkpoint('pilots')

stage(9, 'Pilots (language, model)', end=True)

In [ ]:
# ================= CELL 10: baseline gate + headroom (§8.1, §8.4) ============
# Read the printed gate. A baseline error rate above ~0.60 means the measurement is
# suspect, not the model: inspect report/validation_pairs_tier1_B0.md before trusting
# anything downstream. Also prints the headroom — the share of errors that involve
# syllabus terms at all, which bounds any possible gain.
stage(10, 'Baseline + validation gate')

run('src/run_matrix.py', 'baseline', '--tier', 'tier1')
checkpoint('baseline')

stage(10, 'Baseline + validation gate', end=True)

In [ ]:
# ======================= CELL 11: Tier-1 tuning (~1 h) =======================
# Every hyperparameter is chosen here and only here, on the development slice, so no
# setting is ever tuned on the data the results are reported from.
stage(11, 'Tier-1 tuning')

run('src/run_matrix.py', 'tune', '--tier', 'tier1')
checkpoint('tune')

stage(11, 'Tier-1 tuning', end=True)

In [ ]:
# =================== CELL 12: Tier-2 matrix — the results (~2-3 h) ===========
# Six decodes plus two ablation decodes; every output-level row and combination is
# free, and gating re-uses decodes already paid for. Then statistics, tables, figures.
stage(12, 'Tier-2 experiment matrix')

run('src/run_matrix.py', 'matrix', '--tier', 'tier2')
checkpoint('matrix_tier2')

stage(12, 'Tier-2 experiment matrix', end=True)

In [ ]:
# ============================== CELL 13: results =============================
stage(13, 'Results')

run('src/status.py')
from IPython.display import Markdown, Image, display
import pathlib
for f in ('report/results_tier2.md', 'report/results_tier3.md'):
    if pathlib.Path(f).exists():
        display(Markdown(pathlib.Path(f).read_text()))
for f in sorted(pathlib.Path('report/figures').glob('*.png')):
    print(f); display(Image(str(f)))

stage(13, 'Results', end=True)
print('\n\n***** SESSION COMPLETE — pick the best system from the table above, '
      'then run the Tier-3 cell in a NEW session *****')

---

## Tier 3 — a separate session

The cell below is **Markdown on purpose**, so a full run skips it. The final confirmation
decodes the complete 5.18-hour test set twice (the baseline and the one best system),
roughly 2–3 hours on a T4, and there is no point running it until the Tier-2 table exists.

**To run it, in a new session:**

1. Attach this notebook's previous output: *Add Data → Notebook Output → this notebook*.
2. Change the cell below from **Markdown to Code** using the cell-type dropdown.
3. Set `best` from `report/results_tier2.md`.
4. `Save Version → Save & Run All`.

Cells 1–12 will re-run, but with the cache restored they finish in minutes because nothing
needs decoding again — **check that cell 2 prints a few thousand restored decodes**. If it
says "no previous checkpoint attached", stop and attach the output first, or the session
will be spent repeating Tier 1 and Tier 2 instead of doing Tier 3.

Setting `best = 'G'` is usually the best value: gating is built on top of M2, so the runner
decodes the baseline **and** M2 on the full test set and then composes G from them for
free — three rows for the price of two decodes.

```python
stage(14, 'Tier 3 — final confirmation')

best = 'G'   # set from report/results_tier2.md: G, M2, M1, M2+M3a or M3a
run('src/run_matrix.py', 'final', '--tier', 'tier3', '--best', best)
checkpoint('final_tier3')
run('src/make_setup_section.py')
run('src/repro.py')
checkpoint('done')

stage(14, 'Tier 3 — final confirmation', end=True)
```